In [1]:
# ensure to run on GPU
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 14137133732468262460
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 21845114880
locality {
  bus_id: 1
  links {
  }
}
incarnation: 12003644884220841850
physical_device_desc: "device: 0, name: NVIDIA L4, pci bus id: 0000:00:03.0, compute capability: 8.9"
xla_global_id: 416903419
]


In [2]:
pip install diffusers

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleDDPM(nn.Module):
    def __init__(self, model, timesteps=1000):
        super().__init__()
        self.model = model
        self.timesteps = timesteps

        betas = torch.linspace(1e-4, 0.02, timesteps)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)

        # register as model buffers to store in GPU
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cumprod", alphas_cumprod)
        self.register_buffer("sqrt_alphas_cumprod", torch.sqrt(alphas_cumprod))
        self.register_buffer("sqrt_one_minus_alphas_cumprod", torch.sqrt(1.0 - alphas_cumprod))

    # obtain constants for specific time step t
    def extract(self, a, t, x_shape):
        batch_size = t.shape[0]
        out = a.gather(-1, t)
        return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))

    # forward process
    def q_sample(self, x_start, t, noise):
        # x_t = sqrt(alpha_bar) * x_0 + sqrt(1 - alpha_bar) * noise
        sqrt_alphas_cumprod_t = self.extract(self.sqrt_alphas_cumprod, t, x_start.shape)
        sqrt_one_minus_alphas_cumprod_t = self.extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape)
        return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise

    def forward(self, x_start):
        batch_size = x_start.shape[0]
        t = torch.randint(0, self.timesteps, (batch_size,), device=x_start.device).long() # random timestep
        noise = torch.randn_like(x_start) # random noise
        x_t = self.q_sample(x_start, t, noise) # get image x_t
        predicted_noise = self.model(x_t, t).sample # predict noise
        loss = F.mse_loss(predicted_noise, noise) # get MSE loss
        return loss

    # generate new images
    @torch.no_grad()
    def sample(self, image_shape, device):
        x = torch.randn(image_shape, device=device) # pure noise
        for i in reversed(range(self.timesteps)):
            t = torch.full((image_shape[0],), i, device=device, dtype=torch.long)
            predicted_noise = self.model(x, t).sample # predict noise
            alpha = self.extract(self.alphas, t, x.shape)
            alpha_cumprod = self.extract(self.alphas_cumprod, t, x.shape)
            beta = self.extract(self.betas, t, x.shape)
            # add random noise to ensure diversity
            if i > 0: noise = torch.randn_like(x)
            else: noise = torch.zeros_like(x)
            # core: denoising formula
            x = (1 / torch.sqrt(alpha)) * (x - ((1 - alpha) / (torch.sqrt(1 - alpha_cumprod))) * predicted_noise) + torch.sqrt(beta) * noise
        return x

In [4]:
from diffusers import UNet2DModel
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# initialize a simple U-Net
unet = UNet2DModel(
    sample_size=28,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(64, 128),
    down_block_types=("DownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "UpBlock2D"),
)

# DDPM model
device = "cuda" if torch.cuda.is_available() else "cpu"
ddpm = SimpleDDPM(model=unet, timesteps=1000).to(device)
optimizer = Adam(ddpm.parameters(), lr=1e-4)

# load MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# start training
epochs = 5
for epoch in range(epochs):
    for batch_idx, (images, _) in enumerate(dataloader):
        images = images.to(device)
        optimizer.zero_grad()
        loss = ddpm(images)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1} | Loss: {loss.item():.4f}")

print("Training completed!")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
100%|██████████| 9.91M/9.91M [00:01<00:00, 5.56MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 133kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.4MB/s]


Epoch 1 | Loss: 0.0279
Epoch 2 | Loss: 0.0315
Epoch 3 | Loss: 0.0216
Epoch 4 | Loss: 0.0310
Epoch 5 | Loss: 0.0347
Training completed!


In [7]:
import os
from torchvision.utils import save_image
from IPython.display import Image, display

output_dir = "generated_image"
os.makedirs(output_dir, exist_ok=True)

for i in range(100):
    generated_images = ddpm.sample((16, 1, 28, 28), device)
    file_path = os.path.join(output_dir, f"{i+1}.png")
    save_image(
        generated_images,
        file_path,
        nrow=4,
        normalize=True,
        value_range=(-1, 1)
    )
    if (i+1) % 10 == 0: print(f"generated {i+1}/100 images")

generated 10/100 images
generated 20/100 images
generated 30/100 images
generated 40/100 images
generated 50/100 images
generated 60/100 images
generated 70/100 images
generated 80/100 images
generated 90/100 images
generated 100/100 images


In [10]:
!zip -r image_archive2.zip generated_image/

  adding: generated_image/ (stored 0%)
  adding: generated_image/86.png (deflated 1%)
  adding: generated_image/12.png (deflated 2%)
  adding: generated_image/95.png (deflated 2%)
  adding: generated_image/21.png (deflated 2%)
  adding: generated_image/79.png (deflated 1%)
  adding: generated_image/89.png (deflated 1%)
  adding: generated_image/71.png (deflated 2%)
  adding: generated_image/35.png (deflated 2%)
  adding: generated_image/75.png (deflated 1%)
  adding: generated_image/8.png (deflated 2%)
  adding: generated_image/25.png (deflated 2%)
  adding: generated_image/63.png (deflated 2%)
  adding: generated_image/69.png (deflated 2%)
  adding: generated_image/17.png (deflated 1%)
  adding: generated_image/97.png (deflated 3%)
  adding: generated_image/47.png (deflated 1%)
  adding: generated_image/58.png (deflated 1%)
  adding: generated_image/28.png (deflated 2%)
  adding: generated_image/4.png (deflated 2%)
  adding: generated_image/22.png (deflated 1%)
  adding: generated_ima